In [6]:
import pandas as pd

### Load Dataset

In [7]:
df = pd.read_csv("../data/email_evaluation_dataset_isha-bhole.csv")
df.head()

,id,email_text,expected_action,expected_tone
0,1,Newsletter: Top ML papers this week.,ignore,neutral
1,2,Weekly newsletter: Latest AI research updates.,ignore,neutral
2,3,Automatic reply: Out of office till 2025-12-20.,ignore,neutral
3,4,Scholarship application status: Shortlisted. N...,ignore,neutral
4,5,"Payment reminder: INR INR 10,602 outstanding. ...",notify,polite


###  Create a cleaned text column
Before applying or analysing rules, a clean_text column is created from email_text by lowercasing and removing non‑alphabet characters.

In [8]:
df["clean_text"] = (
    df["email_text"]
      .astype(str)
      .str.lower()
      .str.replace("[^a-zA-Z ]", "", regex=True)
)
df.head()

,id,email_text,expected_action,expected_tone,clean_text
0,1,Newsletter: Top ML papers this week.,ignore,neutral,newsletter top ml papers this week
1,2,Weekly newsletter: Latest AI research updates.,ignore,neutral,weekly newsletter latest ai research updates
2,3,Automatic reply: Out of office till 2025-12-20.,ignore,neutral,automatic reply out of office till
3,4,Scholarship application status: Shortlisted. N...,ignore,neutral,scholarship application status shortlisted nex...
4,5,"Payment reminder: INR INR 10,602 outstanding. ...",notify,polite,payment reminder inr inr outstanding due jan


### Define the email_assistant logic
Here a function email_assistant(email_text) is defined that reads the email text, converts it to lowercase, and looks for simple keywords.
Based on these keywords it returns a pair (action, tone), for example ("respond", "urgent") for deadlines or payments, and ("ignore", "neutral") for marketing emails.

In [24]:
def email_assistant(clean_text):
    text = str(clean_text).lower()
    if any(k in text for k in ["urgent", "deadline", "submit", "due", "payment", "invoice", "overdue"]):
        return "respond", "urgent"
    if any(k in text for k in ["suspicious", "security alert", "phishing", "account suspended"]):
        return "notify", "urgent"
    if any(k in text for k in ["sale", "offer", "discount", "free", "lottery", "win"]):
        return "ignore", "neutral"
    if any(k in text for k in ["mentor", "professor", "meeting", "project", "thesis", "assignment"]):
        return "respond", "polite"
    return "respond", "neutral"

### Generate predictions for every email
The output is split into two new columns, predicted_action and predicted_tone, which represent what the rule‑based assistant would do for each email.

In [26]:
df["predicted_action"], df["predicted_tone"] = zip(
    *df["clean_text"].apply(email_assistant)
)

df[["clean_text", "expected_action", "predicted_action",
    "expected_tone", "predicted_tone"]].head()

,clean_text,expected_action,predicted_action,expected_tone,predicted_tone
0,newsletter top ml papers this week,ignore,respond,neutral,neutral
1,weekly newsletter latest ai research updates,ignore,respond,neutral,neutral
2,automatic reply out of office till,ignore,respond,neutral,neutral
3,scholarship application status shortlisted nex...,ignore,respond,neutral,neutral
4,payment reminder inr inr outstanding due jan,notify,respond,polite,urgent


### Mark which predictions are correct
Two Boolean columns, action_correct and tone_correct, are created by comparing predictions with the expected labels.
For each row these columns are True when the assistant’s output matches the human label, and False when it is wrong.

In [27]:
df["action_correct"] = df["predicted_action"] == df["expected_action"]
df["tone_correct"] = df["predicted_tone"] == df["expected_tone"]

df[["expected_action", "predicted_action", "action_correct"]].head()
df[["expected_tone", "predicted_tone", "tone_correct"]].head()

,expected_tone,predicted_tone,tone_correct
0,neutral,neutral,True
1,neutral,neutral,True
2,neutral,neutral,True
3,neutral,neutral,True
4,polite,urgent,False


### Calculate overall accuracy
This tells how often the assistant made the right decision across the whole evaluation dataset.

In [32]:
action_accuracy = df["action_correct"].mean() * 100
tone_accuracy = df["tone_correct"].mean() * 100

print("Action accuracy (%):", action_accuracy)
print("Tone accuracy (%):", tone_accuracy)

Action accuracy (%): 56.99999999999999
Tone accuracy (%): 67.0


### Error analysis
Rows where action_correct or tone_correct are False are filtered into separate tables for mistakes.

In [29]:
errors = df[df["action_correct"] == False][
    ["clean_text", "expected_action", "predicted_action"]
]
num_action_errors = len(errors)
print("Number of action prediction errors:", num_action_errors)
errors.head(20)

Number of action prediction errors: 43


,clean_text,expected_action,predicted_action
0,newsletter top ml papers this week,ignore,respond
1,weekly newsletter latest ai research updates,ignore,respond
2,automatic reply out of office till,ignore,respond
3,scholarship application status shortlisted nex...,ignore,respond
4,payment reminder inr inr outstanding due jan,notify,respond
6,server maintenance completed all systems normal,ignore,respond
7,thesis review meeting confirmed for jan prepa...,notify,respond
12,thesis review meeting confirmed for prepare s...,notify,respond
16,project extension approved till jan update pr...,ignore,respond
17,could you clarify the dataset format for crop ...,ignore,respond


In [30]:
tone_errors = df[df["tone_correct"] == False][
    ["clean_text", "expected_tone", "predicted_tone"]
]
num_tone_errors = len(tone_errors)
print("Number of tone prediction errors:", num_tone_errors)
tone_errors.head(20)

Number of tone prediction errors: 33


,clean_text,expected_tone,predicted_tone
4,payment reminder inr inr outstanding due jan,polite,urgent
10,quick question can you review my code changes ...,polite,neutral
11,need help with nltk stemming any recommendations,polite,neutral
13,reminder team meeting tomorrow at am please c...,urgent,polite
14,quick question can you review my code changes ...,polite,neutral
16,project extension approved till jan update pr...,neutral,polite
18,urgent meeting rescheduled to today pm join z...,polite,urgent
20,reminder team meeting tomorrow at pm ist plea...,urgent,polite
23,project sync scheduled for at pm agenda atta...,neutral,polite
24,grades updated for ml course check portal for ...,polite,neutral


### Save the evaluated dataset

In [31]:
out_path = "../data/milestone2_output_isha-bhole.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: ../data/milestone2_output_isha-bhole.csv


### Reflection

##### 1. Which type of emails were hardest to classify?
The hardest emails were the “in‑between” ones that were partly important but not clearly urgent, such as general project updates, friendly check‑ins, and information emails that sometimes needed a reply and sometimes did not. Many promotional or newsletter‑style emails were also confusing when they mentioned real tasks (like conferences or events) along with marketing language.

##### 2. Why did your rules fail in some cases?
The rules failed because they only look for a few fixed keywords and cannot understand full sentence meaning or context. For example, if the text contains words like “meeting” or “deadline” the rule always predicts respond, even when the email is just a reminder or summary and does not really need an action. The rules also cannot understand tone correctly when there is no clear word like “urgent”, so they often guess neutral or polite even when the message feels more serious.

##### 3. How could an LLM improve this process?
An LLM could read the whole email and understand the intent, such as whether the sender is actually asking for a reply or only sharing information. It can also detect subtle signals of urgency and emotion instead of depending only on a small keyword list. This would reduce both action errors and tone errors, especially on borderline cases where simple rules are not enough.

In [45]:
pip install langsmith

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
from langsmith import Client
client = Client()

In [47]:
#Define the Judge Prompt

judge_prompt = """You are an evaluator. Compare the model output with the ideal answer.

Check:
1. Action correctness
2. Tone correctness

Give score:
1 = correct
0 = incorrect
"""

In [48]:
# Run agent + Judge
def evaluate(agent_output, ideal_action, ideal_tone):
    if (
        agent_output["action"] == ideal_action 
        and agent_output["tone"] == ideal_tone
    ):
        return 1
    else:
        return 0

In [49]:
agent_output ={
    "action":"notify",
    "tone":"urgent"
}

In [50]:
ideal_action = "notify"
ideal_tone = "urgent"

In [51]:
score = evaluate(agent_output, ideal_action, ideal_tone)
score

1